In [ ]:
import multiprocessing as mp
import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Callable, final, override

import gdown
import lightning as L
import pandas as pd
import timm
import timm.data
import torch
import torch.nn as nn
import torchmetrics
import wandb
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import WandbLogger
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchmetrics import classification

os.environ["WANDB_NOTEBOOK_NAME"] = "baseline.ipynb"

SEED = 67
L.seed_everything(SEED, workers=True)
torch.set_float32_matmul_precision("high")


@final
class Comp:
    name = [
        "glass-insulator",
        "lightning-rod-suspension",
        "polymer-insulator-upper-shackle",
        "vari-grip",
        "yoke-suspension",
    ]
    id = {name: i for i, name in enumerate(name)}

    @staticmethod
    def to_id(name: str):
        return Comp.id[name]

    @staticmethod
    def to_name(id: int):
        return Comp.name[id]


@final
class Stat:
    @staticmethod
    def to_id(name: str):
        return name == "bad"

    @staticmethod
    def to_name(id: bool):
        return "bad" if id else "good"


@final
@dataclass(frozen=True, slots=True)
class Config:
    epochs_phase1: int = 5  # Phase 1: 凍結主幹，僅訓練 Head
    epochs_phase2: int = 15  # Phase 2: 解凍主幹最後幾層微調
    lr: float = 1e-3  # Phase 1 使用較大 LR
    weight_decay: float = 1e-2
    precision: str = "16-mixed"


@final
@dataclass(frozen=True, slots=True)
class DataConfig:
    data_dir: Path = Path("data")
    train_dir: Path = data_dir / "train_dataset"
    test_dir: Path = data_dir / "test_dataset"
    image_size: int = 512
    batch_size: int = 32
    test_batch_size: int = 128  # maximum batch size that can fit in GPU memory
    num_workers: int = mp.cpu_count()
    pin_memory: bool = True
    persistent_workers: bool = True
    prefetch_factor: int = 2
    n_folds: int = 5


tcfg = Config()
dcfg = DataConfig()

In [ ]:
@final
class TrainingDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: Callable):
        super().__init__()
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    @override
    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, bool]:
        row = self.df.iloc[index]
        img = Image.open(row["path"]).convert("RGB")
        img = self.transform(img)
        return img, row["comp"], row["stat"]

In [ ]:
@final
class InspladDataModule(L.LightningDataModule):
    def __init__(
        self, train_transforms: Callable, test_transforms: Callable, fold_idx: int
    ):
        super().__init__()
        self.train_transforms = train_transforms
        self.test_transforms = test_transforms
        self.fold_idx = fold_idx

        self.train_ds: TrainingDataset
        self.val_ds: TrainingDataset

    @override
    def prepare_data(self):
        train_file_id = r"14B3Jsj4DzoCrMC4YXlXEp0ej_reMgWkq"
        train_checksum = r"md5:449e4617aefa0d9c9d059e21c38b32f5"
        test_file_id = r"1RhPBNwWxRYK0M8UvQcBBnVSpPzsEF3xn"
        test_checksum = r"md5:5edc01fa26e9563449aa7e7885242e71"

        gdown.cached_download(
            id=train_file_id,
            path=f"{dcfg.train_dir}.zip",
            hash=train_checksum,
        )
        gdown.cached_download(
            id=test_file_id,
            path=f"{dcfg.test_dir}.zip",
            hash=test_checksum,
        )
        gdown.extractall(f"{dcfg.train_dir}.zip")
        gdown.extractall(f"{dcfg.test_dir}.zip")

    @override
    def setup(self, stage: str):
        if stage == "fit":
            # Unlovely implementation, but I believe this is the only way to do cross validations with DataModule
            full_df = pd.DataFrame(
                [
                    (p, Comp.to_id(p.parents[1].stem), Stat.to_id(p.parent.stem))
                    for p in sorted(dcfg.train_dir.rglob("*.jpg"))
                ],
                columns=["path", "comp", "stat"],
            )

            skf = StratifiedKFold(
                n_splits=dcfg.n_folds, shuffle=True, random_state=SEED
            )
            stratify = full_df["comp"] * 2 + full_df["stat"]
            train_idx, val_idx = list(skf.split(full_df, stratify))[self.fold_idx]

            train_df = full_df.iloc[train_idx]
            val_df = full_df.iloc[val_idx]
            self.train_ds = TrainingDataset(train_df, self.train_transforms)
            self.val_ds = TrainingDataset(val_df, self.test_transforms)

    @override
    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=dcfg.batch_size,
            shuffle=True,
            num_workers=dcfg.num_workers,
            pin_memory=dcfg.pin_memory,
            persistent_workers=dcfg.persistent_workers,
            prefetch_factor=dcfg.prefetch_factor,
        )

    @override
    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=dcfg.test_batch_size,
            shuffle=False,
            num_workers=dcfg.num_workers,
            pin_memory=dcfg.pin_memory,
            persistent_workers=dcfg.persistent_workers,
            prefetch_factor=dcfg.prefetch_factor,
        )

    # def test_dataloader(self):
    #     return DataLoader(
    #         self.test_ds,
    #         batch_size=dcfg.test_batch_size,
    #         shuffle=False,
    #         num_workers=dcfg.num_workers,
    #         pin_memory=dcfg.pin_memory,
    #     )

In [ ]:
@final
class MultiHeadDino(L.LightningModule):
    def __init__(self, dino: str = "vit_large_patch16_dinov3_qkvb.lvd1689m"):
        super().__init__()
        self.save_hyperparameters(
            {
                "dino": dino,
                "epochs_phase1": tcfg.epochs_phase1,
                "epochs_phase2": tcfg.epochs_phase2,
                "lr": tcfg.lr,
                "weight_decay": tcfg.weight_decay,
                "image_size": dcfg.image_size,
                "batch_size": dcfg.batch_size,
                "test_batch_size": dcfg.test_batch_size,
                "n_folds": dcfg.n_folds,
                "seed": SEED,
            }
        )
        # 建議使用 pos_weight，你可以先設個通用值，或針對 5 個器材各給一個權重
        self.criterion = nn.BCEWithLogitsLoss()
        self.backbone = timm.create_model(
            dino, pretrained=True, num_classes=0, dynamic_img_size=True
        )

        for param in self.backbone.parameters():
            param.requires_grad = False

        embed_dim = self.backbone.num_features
        self.head = nn.Linear(embed_dim, len(Comp.name))

        self.val_metrics = {
            c: torchmetrics.MetricCollection(
                {
                    # "recall@p92": classification.BinaryRecallAtFixedPrecision(0.92),
                    "ap": classification.BinaryAveragePrecision(),
                },
                prefix=f"{Comp.to_name(c)}_",
            )
            for c in Comp.id.values()
        }

    @override
    def forward(self, img: torch.Tensor, comp: torch.Tensor):
        feature = self.backbone(img)
        all_logits = self.head(feature)

        batch_size = img.size(0)
        batch_indices = torch.arange(batch_size)

        selected_logits = all_logits[batch_indices, comp]
        return selected_logits

    @override
    def training_step(
        self, batch: tuple[torch.Tensor, torch.Tensor, torch.Tensor], batch_idx: int
    ):
        img, comp, stat = batch
        pred = self(img, comp)

        loss = self.criterion(pred, stat.float())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    @override
    def validation_step(
        self, batch: tuple[torch.Tensor, torch.Tensor, torch.Tensor], batch_idx: int
    ):
        img, comp, stat = batch
        pred = self(img, comp)

        loss = self.criterion(pred, stat.float())
        self.log("val_loss", loss, batch_size=stat.shape[0], sync_dist=True)
        for c, m in self.val_metrics.items():
            mask = comp == c
            m.update(pred[mask], stat[mask])

    @override
    def on_validation_epoch_end(self):
        for m in self.val_metrics.values():
            self.log_dict(m.compute())
            m.reset()

    @override
    def configure_optimizers(self):
        # 最佳實踐：使用 AdamW 搭配 Cosine Annealing
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=tcfg.lr, weight_decay=tcfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=tcfg.epochs_phase1
        )
        return [optimizer], [scheduler]

    def build_transforms(self):
        data_config = timm.data.resolve_model_data_config(self.backbone)
        data_config["input_size"] = (3, dcfg.image_size, dcfg.image_size)
        train_transform = timm.data.create_transform(**data_config, is_training=True)
        test_transform = timm.data.create_transform(**data_config, is_training=False)
        return train_transform, test_transform

In [ ]:
# import matplotlib.pyplot as plt
# from torchvision.transforms import v2

# dm = InspladDataModule(
#     v2.Compose(
#         [
#             v2.Resize((dcfg.image_size, dcfg.image_size)),
#             v2.ToImage(),
#             v2.ToDtype(torch.float32, scale=True),
#         ]
#     ),
#     lambda x: x,
#     0,
# )
# dm.prepare_data()
# dm.setup("fit")
# loader = dm.train_dataloader()
# for img, comp, stat in loader:
#     print(img.shape, comp, stat)
#     print(img.dtype, comp.dtype, stat.dtype)

#     fig, axes = plt.subplots(3, 4, figsize=(12, 12))
#     for i, ax in enumerate(axes.flat):
#         if i < img.shape[0]:
#             ax.imshow(img[i].permute(1, 2, 0).cpu().numpy())
#             title = f"{Comp.to_name(comp[i].item())}\n{Stat.to_name(stat[i].item())}"
#             color = "orange" if stat[i].item() else "black"
#             ax.set_title(title, color=color)
#             ax.axis("off")
#     plt.tight_layout()
#     plt.show()
#     break

# exit
# exit
# exit

In [ ]:
group_name = datetime.now().strftime("%m%d_%H%M")

models = []
for fold_idx in range(dcfg.n_folds):
    model = MultiHeadDino()
    dm = InspladDataModule(*model.build_transforms(), fold_idx)

    wandb.finish()
    wandb_logger = WandbLogger(
        project="InsPLAD",
        group=group_name,
        name=f"fold-{fold_idx}",
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        min_delta=0.0,
    )
    trainer = L.Trainer(
        logger=wandb_logger,
        callbacks=[early_stop],
        max_epochs=tcfg.epochs_phase1 + tcfg.epochs_phase2,
        accelerator="auto",
        devices="auto",
        precision=tcfg.precision,
        enable_checkpointing=False,
        enable_model_summary=False,
        num_sanity_val_steps=0,
        # log_every_n_steps=10,
    )
    trainer.fit(model, datamodule=dm)
    models.append(model)

In [ ]:
# testing